1st preparing the dataset

In [ ]:
import datasets
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = datasets.load_dataset("glue", "mrpc")
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_function(data):
    return tokenizer(data["sentence1"], data["sentence2"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

2nd defining model and train it

In [3]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

training_arg = TrainingArguments("test-trainer")
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
trainer = Trainer(model,
                  training_arg,
                  train_dataset = tokenized_datasets["train"],
                  eval_dataset = tokenized_datasets["validation"],
                  data_collator = data_collator,
                  processing_class = tokenizer)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [4]:
trainer.train()

Step,Training Loss
500,0.550714
1000,0.330400


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1377, training_loss=0.3776280687785097, metrics={'train_runtime': 212.5619, 'train_samples_per_second': 51.768, 'train_steps_per_second': 6.478, 'total_flos': 405114969714960.0, 'train_loss': 0.3776280687785097, 'epoch': 3.0})

3rd building a metric computing function
- understanding the fundamental

In [ ]:
!pip install evaluate

In [9]:
predictions = trainer.predict(tokenized_datasets["validation"])
print(predictions.predictions.shape, predictions.label_ids.shape)
"""
(408, 2) : #elements in the daset and #labels
(408,) : #true labels
"""

import numpy as np
preds = np.argmax(predictions.predictions, axis=-1) #arg max to clearly see the prediction and compare w labels

"""
with the evaluate.load() function. The object returned has a compute() method we can use to do the metric calculation
"""
import evaluate
metric = evaluate.load("glue", "mrpc")
metric.compute(predictions=preds, references=predictions.label_ids)

(408, 2) (408,)


{'accuracy': 0.8627450980392157, 'f1': 0.9047619047619048}

- compute_metric function

In [14]:
def compute_metric(eval_pred):
  logits, labels = eval_pred
  metric = evaluate.load("glue", "mrpc")
  preds = np.argmax(logits, axis=-1)
  return metric.compute(predictions=preds, references=predictions.label_ids)

#new trainer
new_training_args = TrainingArguments("test-trainer", eval_strategy="epoch")
new_trainer = Trainer(model,
                  new_training_args,
                  train_dataset = tokenized_datasets["train"],
                  eval_dataset = tokenized_datasets["validation"],
                  compute_metrics = compute_metric,
                  data_collator = data_collator,
                  processing_class = tokenizer)
new_trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.780986,0.825980,0.872987
2,0.292076,0.650438,0.860294,0.901554
3,0.163320,0.765039,0.848039,0.896321


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1377, training_loss=0.20566941138705916, metrics={'train_runtime': 243.2286, 'train_samples_per_second': 45.241, 'train_steps_per_second': 5.661, 'total_flos': 405114969714960.0, 'train_loss': 0.20566941138705916, 'epoch': 3.0})

Advanced Training settings
- Mixed Precision Training: Use fp16=True in your training arguments for faster training and reduced memory usage
- Gradient Accumulation: For effective larger batch sizes when GPU memory is limited
- Learning Rate Scheduling: The Trainer uses linear decay by default, but you can customize this

In [ ]:
#Mixed Precision Training
training_args = TrainingArguments(
    "test-trainer",
    eval_strategy="epoch",
    fp16=True,  # Enable mixed precision
)

#Gradient Accumulation
training_args = TrainingArguments(
    "test-trainer",
    eval_strategy="epoch",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # Effective batch size = 4 * 4 = 16
)

#Learning Rate Scheduling
training_args = TrainingArguments(
    "test-trainer",
    eval_strategy="epoch",
    learning_rate=2e-5,
    lr_scheduler_type="cosine",  # Try different schedulers
)